In [ ]:
# %% [markdown]
# # CodeBook: Social Media Data Analysis (Pure Python)
# ### *Data Science Internship Project*
# 
# **Project Goal:** Analyze user connections and build recommendation engines 
# using **only built-in Python modules** (no Pandas, NumPy, or Scikit-learn).

import json

# %% [markdown]
# ## 1. Data Loading
# We start by loading the raw data dump from CodeBook.

def load_data(filename):
    """Loads JSON data from a local file."""
    with open(filename, "r") as f:
        return json.load(f)

# Initial load
data = load_data("data/raw_data.json")

# %% [markdown]
# ## 2. Data Cleaning & Structuring
# The raw data contains missing names, duplicate friends, and inactive accounts. 
# We must clean this before analysis.

def clean_data(data):
    # 1. Remove users with missing names
    data["users"] = [u for u in data["users"] if u["name"].strip()]
    
    # 2. Deduplicate friend lists
    for user in data["users"]:
        user['friends'] = list(set(user['friends']))
        
    # 3. Remove inactive users (no friends AND no liked pages)
    data['users'] = [u for u in data['users'] if u['friends'] or u['liked_pages']]

    # 4. Deduplicate pages by ID
    unique_pages = {p['id']: p for p in data['pages']}
    data['pages'] = list(unique_pages.values())
    
    return data

cleaned_data = clean_data(data)
print("Data cleaned successfully!")

# %% [markdown]
# ## 3. Feature: 'People You May Know'
# This algorithm suggests new connections based on **Mutual Friend counts**. 
# Higher mutual friends = Higher priority.

def find_people_you_may_know(user_id, data):
    # Map users to their friend sets for O(1) lookup
    user_friends = {u["id"]: set(u["friends"]) for u in data["users"]}
    
    if user_id not in user_friends:
        return []

    direct_friends = user_friends[user_id]
    suggestions = {}

    for friend_id in direct_friends:
        # Check friends of my friends
        for mutual in user_friends.get(friend_id, []):
            if mutual != user_id and mutual not in direct_friends:
                suggestions[mutual] = suggestions.get(mutual, 0) + 1
                
    # Sort by number of mutual friends
    sorted_suggestions = sorted(suggestions.items(), key=lambda x: x[1], reverse=True)
    return [uid for uid, count in sorted_suggestions]

# Example usage
print(f"Suggestions for User 1: {find_people_you_may_know(1, cleaned_data)}")

# %% [markdown]
# ## 4. Feature: 'Pages You Might Like'
# Implementing **Collaborative Filtering**: If two users like the same pages, 
# they likely share interests. We recommend pages liked by "similar" users.

def find_pages_you_might_like(user_id, data):
    user_pages = {u['id']: set(u['liked_pages']) for u in data['users']}
    
    if user_id not in user_pages:
        return []
        
    target_likes = user_pages[user_id]
    recommendations = {}

    for other_id, other_likes in user_pages.items():
        if other_id == user_id:
            continue
            
        # Calculate similarity score (count of shared pages)
        shared = target_likes.intersection(other_likes)
        score = len(shared)
        
        if score > 0:
            for page in other_likes:
                if page not in target_likes:
                    recommendations[page] = recommendations.get(page, 0) + score

    # Sort pages by recommendation weight
    sorted_pages = sorted(recommendations.items(), key=lambda x: x[1], reverse=True)
    return [pid for pid, weight in sorted_pages]

# Example usage
print(f"Page Recommendations for User 1: {find_pages_you_might_like(1, cleaned_data)}")